## Note

This notebook builds developmental similarity models and tests brain-behavior similarity
using the ROI time series extracted by *1_extract_timecourse_ssROI_DM.py* and
*1_extract_timecourse_ssROI_TP.py*.

The analysis here uses *Despicable Me* (DM) as an example. 

The same pipeline is applied to *The Present* (TP) by changing the input files.


## 1. construct hypothesis-based developmental model

In [ ]:
import numpy as np
import pandas as pd
import itertools
import os
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from scipy.io import savemat

def intersubject_distance(data, outfile_prefix):
    """
    Compute static pairwise intersubject similarity matrices based on different maturational hypotheses.
    """
    n = data.shape[0]
    maxval = np.max(data)
    nn = np.zeros((n, n))
    annakmin = np.zeros((n, n))
    annakmaxminmax = np.zeros((n, n))
    
    combs = itertools.combinations(range(n), 2)
    for i, j in combs:
        diff = abs(data[i] - data[j])
        #nearest-neighbor: similar abilities similar activation
        nn[i, j] = nn[j, i] = maxval - diff
        #consistent high (convergence) - higher abilities similar activation
        annakmin[i, j] = annakmin[j, i] = min(data[i], data[j])
        #variable-high (divergence) - higher abilities more heterogeneous activation
        annakmaxminmax[i, j] = annakmaxminmax[j, i] = maxval - max(data[i], data[j])
    
    np.save(outfile_prefix + '_NN.npy', nn)
    np.save(outfile_prefix + '_annakmin.npy', annakmin)
    np.save(outfile_prefix + '_annakmaxminmax.npy', annakmaxminmax)
    
    #also save into .mat format if needed
        
    # Save as .mat
    savemat(outfile_prefix + '_NN.mat',
            {'NN': nn})

    savemat(outfile_prefix + '_annakmin.mat',
            {'annakmin': annakmin})

    savemat(outfile_prefix + '_annakmaxminmax.mat',
            {'annakmaxminmax': annakmaxminmax})
    
    return {
        'NN': nn,
        'AnnaKmin': annakmin,
        'AnnaKmaxminmax': annakmaxminmax
    }


# Load and filter data
df = pd.read_csv('~/movieDM_subinfo.csv')    
used_df = df[['ID', 'MRI_Track,Age_at_Scan','FD_mean_DM','MRI_Track,Scan_Location','Basic_Demos,Sex','EHQ,EHQ_Total']].dropna()
print("N =", len(used_df)) 
os.makedirs('~/5_movieDM_ISRSA/2_age_model/', exist_ok=True)

subject_ids = used_df['ID'].astype(str).values

# Use Scan_Location only for one-hot encoding
encoder = OneHotEncoder(drop='first', sparse_output=False)
X_site = encoder.fit_transform(used_df[['MRI_Track,Scan_Location']])

# --------- regress age ------
y_age = used_df['MRI_Track,Age_at_Scan'].values

# Combine numeric covariates (incl. Sex) with encoded site
X_num_age = used_df[['FD_mean_DM', 'EHQ,EHQ_Total', 'Basic_Demos,Sex']].values
X_age = np.hstack([X_num_age, X_site])

# Residualize 
reg_age = LinearRegression().fit(X_age, y_age)
residuals_age = y_age - reg_age.predict(X_age)

# Compute intersubject similarity using residuals
similarity_matrices = intersubject_distance(
    residuals_age,
    outfile_prefix='~/5_movieDM_ISRSA/2_age_model/ageresid_similarity'
)

## 2. IS-RSA analysis

In [ ]:
import numpy as np
from scipy.stats import pearsonr
from scipy.spatial.distance import squareform
from sklearn.utils import resample
from itertools import combinations
import itertools
import pandas as pd
from tqdm import tqdm
import os
from glob import glob

# Load subject info & match subjects with age model
df = pd.read_csv('~/movieDM_subinfo.csv')

all_subject_ids = df['ID'].astype(str).values  
print("Whole sample N =", len(all_subject_ids))

used_df = df[['ID', 'MRI_Track,Age_at_Scan','FD_mean_DM','MRI_Track,Scan_Location','Basic_Demos,Sex','EHQ,EHQ_Total']].dropna()

used_subject_ids = used_df['ID'].astype(str).values 
print("Used sample N =", len(used_subject_ids))

id_to_index = {subj_id: i for i, subj_id in enumerate(all_subject_ids)}
matching_indices = [id_to_index[sid] for sid in used_subject_ids]
print("Matching indices N =", len(matching_indices))

# Load similarity matrices
similarities = {
    'Age_resid_NN': np.load("~/5_movieDM_ISRSA/2_age_model/ageresid_similarity_NN.npy"),
    'Age_resid_Con_AnnaKmin': np.load("~/5_movieDM_ISRSA/2_age_model/ageresid_similarity_annakmin.npy"),
    'Age_resid_Div_AnnaKmaxminmax': np.load("~/5_movieDM_ISRSA/2_age_model/ageresid_similarity_annakmaxminmax.npy"),
}

base_dir = "5_movieDM_ISRSA/1_sstop10perc_timecourse/"
networks = ["Whole_CoreLanguage"]  

all_results_rows = []
bootstrap_rows = []

for net in networks:

    print("Processing network:", net)

    # === Step a: Load data ===
    net_folder = os.path.join(base_dir, net)

    npy_files = sorted(glob(os.path.join(net_folder, f'movieDM_{net}_sstop10perc_ROI_meantimecourse.npy')))
    if len(npy_files) == 0:
        raise FileNotFoundError(f"No .npy found in {net_folder}")
    data_path = npy_files[0]
    print("Loading:", data_path)

    data = np.load(data_path)
    print("data.shape =", data.shape)  # expected: (750, 10, 1415)

    # Step 4: Subset the subjects with behavioral info avialbe (N=1411)
    aligned_data = data[:, :, matching_indices]
    print("aligned_data shape:", aligned_data.shape)  # (750, 10, 1411)
    NTR, NROI, NSUBJ = aligned_data.shape

    # === Step b: Compute ISC (pairwise correlation for whole network) ===
    aligned_data_mean = np.mean(aligned_data, axis=1)  # shape: (T, NSUBJ)
    print("aligned_data_mean shape:", aligned_data_mean.shape)

    isc_matrices = np.zeros((NSUBJ, NSUBJ))

    for i, j in itertools.combinations(range(NSUBJ), 2):
        r = np.corrcoef(aligned_data_mean[:, i], aligned_data_mean[:, j])[0, 1]
        isc_matrices[i, j] = isc_matrices[j, i] = r

    # dealing with nan values
    isc_matrices = np.nan_to_num(isc_matrices, nan=0.0)
    print("isc_matrices shape:", isc_matrices.shape)

    # === Step c: Correlate ISC with similarity matrices (RSA) ===
    rsa_results = {}
    for model, sim_matrix in similarities.items():
        sim_vector = squareform(sim_matrix, checks=False)
        isc_vector = squareform(isc_matrices, checks=False)
        r, _ = pearsonr(sim_vector, isc_vector)
        rsa_results[model] = r

    # === Step d: Permutation test to assign p-values ===
    n_permutations = 5000
    rng = np.random.default_rng(seed=42)

    rsa_pvalues = {}
    for model, sim_matrix in similarities.items():
        sim_vector = squareform(sim_matrix, checks=False)
        isc_vector = squareform(isc_matrices, checks=False)
        empirical_r = rsa_results[model]
        null_dist = []

        print(f"Permutation for {net} | {model}")
        for _ in tqdm(range(n_permutations)):
            perm = rng.permutation(NSUBJ)
            perm_sim = sim_matrix[perm][:, perm]
            perm_vector = squareform(perm_sim, checks=False)
            r, _ = pearsonr(perm_vector, isc_vector)
            null_dist.append(r)

        p = (np.sum(np.abs(null_dist) >= np.abs(empirical_r)) + 1) / (n_permutations + 1)
        rsa_pvalues[model] = p

    # === Step e: Collect results ===
    for model in similarities:
        all_results_rows.append({
            "Network": net,
            "Model_name": model,
            "RSA_rvalues": rsa_results[model],
            "RSA_pvalues": rsa_pvalues[model]
        })
        
    # ============================================================
    # Purpose: bootstrap pairwise comparisons between significant models
    # ============================================================
    sig_models = [m for m in similarities if rsa_pvalues[m] < 0.05]

    if len(sig_models) >= 2:
        
        for m1, m2 in combinations(sig_models, 2):
            print(f"Bootstrapping {net} | models {m1} vs {m2}")

            sim_vec1 = squareform(similarities[m1], checks=False)
            sim_vec2 = squareform(similarities[m2], checks=False)
            isc_vector = squareform(isc_matrices, checks=False)
            
            # ------------------------------------------------------------
            # Bootstrap Δz (Fisher-transformed correlation difference)
            # ------------------------------------------------------------

            boot_delta_z = []

            for _ in range(1000):
                idx = resample(np.arange(len(isc_vector)))  # resample edges
                
                r1, _ = pearsonr(isc_vector[idx], sim_vec1[idx])
                r2, _ = pearsonr(isc_vector[idx], sim_vec2[idx])
                
                if not np.isnan(r1) and not np.isnan(r2):
                    
                    # avoid infinite values
                    r1 = np.clip(r1, -0.999999, 0.999999)
                    r2 = np.clip(r2, -0.999999, 0.999999)
                    
                    z1 = np.arctanh(r1)
                    z2 = np.arctanh(r2)
                    
                    boot_delta_z.append(z1 - z2)

            boot_delta_z = np.array(boot_delta_z)

            if len(boot_delta_z) > 0:
                
                # Mean Δz
                mean_delta_z = np.mean(boot_delta_z)
                
                # 95% percentile CI in z-space
                ci_lower_z = np.percentile(boot_delta_z, 2.5)
                ci_upper_z = np.percentile(boot_delta_z, 97.5)
                
                # Convert CI back to r-space
                ci_lower_r = np.tanh(ci_lower_z)
                ci_upper_r = np.tanh(ci_upper_z)
                
                # Two-tailed bootstrap p-value in z-space
                p_boot = 2 * min(
                    np.mean(boot_delta_z <= 0),
                    np.mean(boot_delta_z >= 0)
                )
                
                winner = m1 if mean_delta_z > 0 else m2

                bootstrap_rows.append({
                    "Network": net,
                    "Model_1": m1,
                    "Model_2": m2,
                    "Winner": winner,
                    "Mean_Delta_z": mean_delta_z,
                    "CI_lower_z": ci_lower_z,
                    "CI_upper_z": ci_upper_z,
                    "CI_lower_r": ci_lower_r,
                    "CI_upper_r": ci_upper_r,
                    "Bootstrap_pvalue": p_boot,
                    "N_boot_kept": len(boot_delta_z),
                })
    else:
        print(f"Skipping {net}: only {len(sig_models)} significant model(s)")

# === Save results ===
out_df = pd.DataFrame(all_results_rows)
output_path = "5_movieDM_ISRSA/movieDM_network_ISRSA_results.csv"
out_df.to_csv(output_path, index=False)
print(f"\nSaved results to {output_path}")

bootstrap_df = pd.DataFrame(bootstrap_rows)
bootstrap_output_path = "5_movieDM_ISRSA/movieDM_network_ISRSA_bootstrap_pairwise_delta.csv"
bootstrap_df.to_csv(bootstrap_output_path, index=False)
print(f"Saved bootstrap pairwise comparison results to {bootstrap_output_path}")
